In [1]:
import os
import pandas as pd

In [2]:
with open('/u/rfechner/data/eic_gsm8k_deduplicated/test.parquet', 'rb') as file:
    dataset = pd.read_parquet(file)

In [3]:
dataset.head(2)

,prompt,correct_answer,incorrect_answer,correct_solution,incorrect_solution,explanation,error_type,wrong_step,source_file,line_idx
0,John decides to start collecting art. He pays ...,67500.0,69750.0,"The first 3 pieces each cost 45000/3=$15,000\n...","The first 3 pieces each cost 45000/3=$15,000\n...",Step 3 adds information that is not mentioned ...,Hallucination,3,data/generated_cases_GSM8K/adding_irrelevant_i...,0
1,Irene earns $500 if she works for 40 hours a w...,700.0,750.0,"If Irene worked 50 hours last week, the total ...","If Irene worked 50 hours last week, the total ...",Step 3 adds information that is not mentioned ...,Hallucination,3,data/generated_cases_GSM8K/adding_irrelevant_i...,1


In [4]:
"""
data = {
    "data_source": data_source,
    "prompt": [{
        "role": "user",
        "content": question
    }],
    "ability": "math",
    "reward_model": {
        "style": "rule",
        "ground_truth": answer
    },
    "extra_info": {
        'problem_type' : problem_type,
        'split': split,
        'index': idx,
        'problem_idx' : problem_idx
    }
}
"""
data_source = pd.Series(['gsm8k_distilled_from_eic_gsm8k'] * len(dataset))
prompt = dataset['prompt'].apply(lambda x: [{'role' : 'user', 'content' : x}])
ability = pd.Series(['math'] * len(dataset))
reward_model = dataset['correct_answer'].apply(lambda x: {'style' : 'rule', 'ground_truth' : x})
extra_info = dataset['line_idx'].apply(lambda x: {'index' : x})

new_dataset = pd.DataFrame({
'data_source' : data_source,
'prompt' : prompt,
'ability' : ability,
'reward_model' : reward_model,
'extra_info' : extra_info
})

In [5]:
new_dataset.head(2)

,data_source,prompt,ability,reward_model,extra_info
0,gsm8k_distilled_from_eic_gsm8k,"[{'role': 'user', 'content': 'John decides to ...",math,"{'style': 'rule', 'ground_truth': 67500.0}",{'index': 0}
1,gsm8k_distilled_from_eic_gsm8k,"[{'role': 'user', 'content': 'Irene earns $500...",math,"{'style': 'rule', 'ground_truth': 700.0}",{'index': 1}


In [6]:
base_path = "/u/rfechner/data/gsm8k_distilled_from_eic_gsm8k"
os.makedirs(base_path, exist_ok=True)
new_dataset.to_parquet(os.path.join(base_path, 'test.parquet'))